In [ ]:

!pip install indic-nlp-library==0.92 --quiet
!pip install datasets==2.20.0 --quiet
!pip install regex --quiet

Could not find platform independent libraries <prefix>

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
Could not find platform independent libraries <prefix>

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
Could not find platform independent libraries <prefix>

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:

import os
import subprocess

RESOURCES_PATH = "./indic_nlp_resources"

if not os.path.exists(RESOURCES_PATH):
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/anoopkunchukuttan/indic_nlp_resources.git",
        RESOURCES_PATH
    ], check=True)
    print(f"indic_nlp_resources cloned to {RESOURCES_PATH}")
else:
    print(f"indic_nlp_resources already present at {RESOURCES_PATH}")

✅ indic_nlp_resources already present at ./indic_nlp_resources


In [ ]:

from indicnlp import common as indic_common
indic_common.set_resources_path(RESOURCES_PATH)

from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
from indicnlp.tokenize import indic_tokenize

print(" indic_nlp_library configured successfully")

test_sentence = "भारत एक महान देश आहे."
norm_factory = IndicNormalizerFactory()
normalizer = norm_factory.get_normalizer("mr")
normalized = normalizer.normalize(test_sentence)
tokens = indic_tokenize.trivial_tokenize(normalized, lang='mr')
print(f"Test sentence : {test_sentence}")
print(f"Normalized    : {normalized}")
print(f"Trivial tokens: {tokens}")

✅ indic_nlp_library configured successfully
Test sentence : भारत एक महान देश आहे.
Normalized    : भारत एक महान देश आहे.
Trivial tokens: ['भारत', 'एक', 'महान', 'देश', 'आहे', '.']


In [ ]:

import sys
from datasets import load_dataset

LANG_CODE    = "marathi"
LANG_SHORT   = "mr"
LANG_SPLIT   = "mar_Deva"         
OUTPUT_FILE  = f"./backend/tokenizers/{LANG_CODE}_corpus.txt"
TARGET_BYTES = 100 * 1024 * 1024    

print(f"Streaming IndicCorpV2 Marathi split until {TARGET_BYTES / 1e6:.0f} MB collected…")

ds = load_dataset(
    "ai4bharat/IndicCorpV2",
    "indiccorp_v2",
    split=LANG_SPLIT,
    streaming=True,
    trust_remote_code=True,
)

collected_bytes = 0
line_count      = 0

with open(OUTPUT_FILE, "w", encoding="utf-8") as fout:
    for example in ds:
        text = example.get("text", "").strip()
        if not text:
            continue
        encoded = (text + "\n").encode("utf-8")
        fout.write(text + "\n")
        collected_bytes += len(encoded)
        line_count       += 1
        if collected_bytes >= TARGET_BYTES:
            break
        if line_count % 200_000 == 0:
            print(f"  {line_count:,} lines | {collected_bytes / 1e6:.1f} MB", flush=True)

print(f"\n✅ Corpus saved → {OUTPUT_FILE}")
print(f"   Lines  : {line_count:,}")
print(f"   Size   : {collected_bytes / 1e6:.2f} MB")

Streaming IndicCorpV2 Marathi split until 105 MB collected…

✅ Corpus saved → ./backend/tokenizers/marathi_corpus.txt
   Lines  : 138,898
   Size   : 104.86 MB


In [ ]:

import re
import unicodedata

# Devanagari Unicode block: U+0900–U+097F
# Tamil Unicode block     : U+0B80–U+0BFF
# We keep Devanagari, digits, basic punctuation, spaces

_DEVA_RE    = re.compile(r'[^\u0900-\u097F\u0966-\u096F\s।\-,.!?;:\'"()]')
_MULTI_SPACE = re.compile(r'\s+')


def clean_marathi(text: str) -> str:
    """Normalize and lightly clean Marathi text."""
    normalizer = norm_factory.get_normalizer("mr")
    text = normalizer.normalize(text)
    text = _DEVA_RE.sub(' ', text)
    text = _MULTI_SPACE.sub(' ', text)
    return text.strip()


sample = "भारताची संस्कृती अत्यंत विविध आणि समृद्ध आहे.  येथे अनेक भाषा बोलल्या जातात!!  123"
print("Raw   :", sample)
print("Clean :", clean_marathi(sample))

Raw   : भारताची संस्कृती अत्यंत विविध आणि समृद्ध आहे.  येथे अनेक भाषा बोलल्या जातात!!  123
Clean : भारताची संस्कृती अत्यंत विविध आणि समृद्ध आहे. येथे अनेक भाषा बोलल्या जातात!!


In [ ]:

MAX_SENTENCES = 500_000 

print(f"Loading up to {MAX_SENTENCES:,} sentences for evaluation…")

sentences = []
with open(OUTPUT_FILE, "r", encoding="utf-8") as fin:
    for line in fin:
        s = clean_marathi(line)
        if len(s) > 5: 
            sentences.append(s)
        if len(sentences) >= MAX_SENTENCES:
            break

print(f" Loaded {len(sentences):,} sentences for evaluation")
print("\nSample sentences:")
for s in sentences[:5]:
    print(" •", s)

Loading up to 500,000 sentences for evaluation…
✅ Loaded 138,851 sentences for evaluation

Sample sentences:
 • ऊती संवर्धन तंत्राचे अनेक उपयोग आहेत. या तंत्राचा उपयोग विशेषकरून जीवशास्त्र व वैद्यकशास्त्रात होतो. वयोवृद्धी, पोषण, लसनिर्मिती, जन्मजात रोगांचे निदान, इंद्रियांचे रोपण, कर्करोग संशोधन व गर्भपोषण या क्षेत्रांत ऊती संवर्धन तंत्र प्रामुख्याने वापरले जाते. पेशींच्या चयापचयावर एखाद्या घटकाचा परिणाम पाहणे, सामान्य किंवा कर्करोगाच्या पेशींवर औषधांचा होणारा परिणाम पाहणे, प्रयोगशाळेत त्वचा तयार करणे इ. बाबी ऊती संवर्धनामुळे शक्य झाल्या आहेत. भाजलेल्या रुग्णाच्या त्वचारोपणासाठी ऊती संवर्धनाद्वारे निर्माण केलेली त्वचा वापरली जाते.
 • शहरातील माध्यमिक विभागाच्या शाळा ३ जानेवारीपर्यंत विद्यार्थ्यांसाठी बंद ठेवण्यात येणार आहेत. मात्र, शिक्षकांना शाळेत जाणे अनिवार्य केले आहे. त्यामुळे शिक्षक, शिक्षकेतर कर्मचाऱ्यांना कोरोना तपासणी सक्तीची केली आहे. सोमवारी ५९८ शिक्षकांची तपासणी करण्यात आली. रविवारी २९८ शिक्षकांची तपासणी झाली होती. त्यामध्ये १८ जणांना कोरोनाची बाधा झाल्याचे आढळले. आतापर्यं

In [ ]:

import re

_PUNCT_RE = re.compile(r'([।॥,;:!?\.\"\'\(\)\[\]\{\}])')

def whitespace_tokenize(text: str) -> list:
    """
    Baseline: split on whitespace, then detach trailing/leading punctuation.
    This ensures 'है।' becomes ['है', '।'] — a cleaner baseline.
    """
    tokens = []
    for raw_tok in text.split():
        expanded = _PUNCT_RE.sub(r' \1 ', raw_tok)
        tokens.extend(t for t in expanded.split() if t)
    return tokens


examples = [
    "भारत एक महान देश आहे।",
    "पुणे महाराष्ट्राची राजधानी आहे।",
    "मराठी भारतातील एक प्रमुख भाषा आहे।",
]

print("=" * 60)
print("Strategy 1: Whitespace Tokenizer — Sample Outputs")
print("=" * 60)
for ex in examples:
    tokens = whitespace_tokenize(clean_marathi(ex))
    print(f"\nInput : {ex}")
    print(f"Tokens: {tokens}")
    print(f"Count : {len(tokens)}")

Strategy 1: Whitespace Tokenizer — Sample Outputs

Input : भारत एक महान देश आहे।
Tokens: ['भारत', 'एक', 'महान', 'देश', 'आहे', '।']
Count : 6

Input : पुणे महाराष्ट्राची राजधानी आहे।
Tokens: ['पुणे', 'महाराष्ट्राची', 'राजधानी', 'आहे', '।']
Count : 5

Input : मराठी भारतातील एक प्रमुख भाषा आहे।
Tokens: ['मराठी', 'भारतातील', 'एक', 'प्रमुख', 'भाषा', 'आहे', '।']
Count : 7


In [ ]:

import regex                       
from typing import Callable, List, Optional


def grapheme_clusters(text: str) -> list:
    """Split text into Unicode extended grapheme clusters (correct for Devanagari conjuncts)."""
    return regex.findall(r'\X', text)


def compute_metrics(
    tokenizer_fn,         
    sentences,             
    strategy_name,         
  
):
    """
    Compute 5 intrinsic tokenization metrics over a list of sentences.

    Metrics
    -------
    fertility   : avg tokens per whitespace-word  (lower = more efficient)
    oov_rate    : fraction of word-types unseen in training vocab
                  (always 0 for Whitespace — every surface form IS a vocab entry)
    nsl         : Normalized Sequence Length = avg(tokens) / avg(chars)
                  captures relative compression
    cpt         : avg Characters Per Token (higher = coarser / less fragmented)
    pcw         : Proportion of Continued Words = fraction of words split into >1 token
                  (always 0 for Whitespace — no word is ever split)
    """

    total_words       = 0
    total_tokens      = 0
    total_chars       = 0
    total_token_chars = 0
    fragmented_words  = 0
    vocab             = set()

    for sent in sentences:
        words  = sent.split()
        tokens = tokenizer_fn(sent)

        total_words       += len(words)
        total_tokens      += len(tokens)
        total_chars       += len(sent)
        total_token_chars += sum(len(t) for t in tokens)
        vocab.update(tokens)

        # PCW: tokenize each whitespace-word individually
        for word in words:
            if len(tokenizer_fn(word)) > 1:
                fragmented_words += 1

    fertility = total_tokens / total_words       if total_words  > 0 else 0.0
    nsl       = total_tokens / total_chars       if total_chars  > 0 else 0.0
    cpt       = total_token_chars / total_tokens if total_tokens > 0 else 0.0
    pcw       = fragmented_words  / total_words  if total_words  > 0 else 0.0

    vocab_size = vocab_size_override if vocab_size_override is not None else len(vocab)

    return {
        "strategy"        : strategy_name,
        "fertility"       : round(fertility, 4),
        "oov_rate"        : round(0.0,       4),   # 0 by construction for Whitespace
        "nsl"             : round(nsl,       6),
        "cpt"             : round(cpt,       4),
        "pcw"             : round(pcw,       4),   # 0 by construction for Whitespace
        "vocab_size"      : vocab_size,
        "total_tokens"    : total_tokens,
        "total_sentences" : len(sentences),
    }

In [ ]:

import time

print("Computing metrics for Strategy 1: Whitespace…")
print(f"  Evaluating on {len(sentences):,} sentences\n")

t0 = time.time()

ws_metrics = compute_metrics(
    tokenizer_fn  = whitespace_tokenize,
    sentences     = sentences,
    strategy_name = "Whitespace (Baseline)",
    vocab_size_override=None,
)

elapsed = time.time() - t0

print("=" * 60)
print(f"  Strategy : {ws_metrics['strategy']}")
print(f"  Language : Marathi (Devanagari)")
print(f"  Time     : {elapsed:.1f}s")
print("=" * 60)
print(f"  Fertility (tokens/word)         : {ws_metrics['fertility']:.4f}")
print(f"  OOV Rate                        : {ws_metrics['oov_rate']:.4f}  [0 by construction]")
print(f"  NSL      (tokens/char)          : {ws_metrics['nsl']:.6f}")
print(f"  CPT      (chars/token)          : {ws_metrics['cpt']:.4f}")
print(f"  PCW      (fragmented word frac) : {ws_metrics['pcw']:.4f}  [0 by construction]")
print(f"  Vocab size (unique token types) : {ws_metrics['vocab_size']:,}")
print(f"  Total tokens produced           : {ws_metrics['total_tokens']:,}")
print(f"  Sentences evaluated             : {ws_metrics['total_sentences']:,}")
print("=" * 60)

Computing metrics for Strategy 1: Whitespace…
  Evaluating on 138,851 sentences

  Strategy : Whitespace (Baseline)
  Language : Marathi (Devanagari)
  Time     : 32.0s
  Fertility (tokens/word)         : 1.1414
  OOV Rate                        : 0.0000  [0 by construction]
  NSL      (tokens/char)          : 0.166052
  CPT      (chars/token)          : 5.1675
  PCW      (fragmented word frac) : 0.1248  [0 by construction]
  Vocab size (unique token types) : 377,413
  Total tokens produced           : 6,483,551
  Sentences evaluated             : 138,851


In [ ]:

from indicnlp.tokenize import indic_tokenize as indic_tok

def word_tokenize_indic(text: str) -> list:
    """
    IndicNLP trivial_tokenize: rule-based, punctuation-aware word tokenizer.
    Handles Devanagari-specific punctuation (।  ॥) and English punct correctly.
    """
    return indic_tok.trivial_tokenize(text, lang='mr')


examples = [
    "भारत एक महान देश आहे।",
    "पुणे महाराष्ट्राची राजधानी आहे।",
    "तुम्ही मराठी बोलता का?",
    "राम, श्याम आणि मोहन तिघेही मित्र आहेत।",
]

print("=" * 60)
print("Strategy 2: Word Tokenizer (IndicNLP) — Sample Outputs")
print("=" * 60)
for ex in examples:
    tokens = word_tokenize_indic(clean_marathi(ex))
    print(f"\nInput : {ex}")
    print(f"Tokens: {tokens}")
    print(f"Count : {len(tokens)}")

Strategy 2: Word Tokenizer (IndicNLP) — Sample Outputs

Input : भारत एक महान देश आहे।
Tokens: ['भारत', 'एक', 'महान', 'देश', 'आहे', '।']
Count : 6

Input : पुणे महाराष्ट्राची राजधानी आहे।
Tokens: ['पुणे', 'महाराष्ट्राची', 'राजधानी', 'आहे', '।']
Count : 5

Input : तुम्ही मराठी बोलता का?
Tokens: ['तुम्ही', 'मराठी', 'बोलता', 'का', '?']
Count : 5

Input : राम, श्याम आणि मोहन तिघेही मित्र आहेत।
Tokens: ['राम', ',', 'श्याम', 'आणि', 'मोहन', 'तिघेही', 'मित्र', 'आहेत', '।']
Count : 9


In [ ]:
    

import regex

def char_tokenize_grapheme(text: str) -> list:
    """
    Character-level tokenizer using Unicode extended grapheme clusters.
    Spaces are excluded (we tokenize the non-space content).
    """
    return [gc for gc in regex.findall(r'\X', text) if gc.strip()]


# Verify on examples — pay close attention to conjuncts
examples_char = [
    ("Simple word",     "भारत"),
    ("With conjunct",   "क्षमा"),
    ("With vowel mark", "दिल्ली"),
    ("Full sentence",   "भारत महान है।"),
]

print("=" * 60)
print("Strategy 3: Character Tokenizer (Grapheme Clusters) — Sample Outputs")
print("=" * 60)
for label, ex in examples_char:
    tokens = char_tokenize_grapheme(ex)
    print(f"\n[{label}]")
    print(f"  Input  : {ex}")
    print(f"  Tokens : {tokens}")
    print(f"  Count  : {len(tokens)}")
    print(f"  (Unicode codepoints would give: {list(ex.replace(' ',''))} — {len(ex.replace(' ',''))} units)")

Strategy 3: Character Tokenizer (Grapheme Clusters) — Sample Outputs

[Simple word]
  Input  : भारत
  Tokens : ['भा', 'र', 'त']
  Count  : 3
  (Unicode codepoints would give: ['भ', 'ा', 'र', 'त'] — 4 units)

[With conjunct]
  Input  : क्षमा
  Tokens : ['क्ष', 'मा']
  Count  : 2
  (Unicode codepoints would give: ['क', '्', 'ष', 'म', 'ा'] — 5 units)

[With vowel mark]
  Input  : दिल्ली
  Tokens : ['दि', 'ल्ली']
  Count  : 2
  (Unicode codepoints would give: ['द', 'ि', 'ल', '्', 'ल', 'ी'] — 6 units)

[Full sentence]
  Input  : भारत महान है।
  Tokens : ['भा', 'र', 'त', 'म', 'हा', 'न', 'है', '।']
  Count  : 8
  (Unicode codepoints would give: ['भ', 'ा', 'र', 'त', 'म', 'ह', 'ा', 'न', 'ह', 'ै', '।'] — 11 units)


In [21]:
# ============================================================
# CELL 14 (FINAL FIX): compute_metrics_v3 with correct PCW
# ============================================================
import unicodedata
import re
import time

def _strip_to_lexical(word: str) -> str:
    """
    Strip all leading and trailing characters that are NOT
    letters or combining marks (vowel signs, halant etc.).
    Uses Unicode category: L* = letters, M* = marks (matras, halant).
    This correctly handles all punctuation including smart quotes,
    em-dash, ellipsis, brackets etc.
    """
    # Strip from left
    start = 0
    while start < len(word):
        cat = unicodedata.category(word[start])
        if cat.startswith('L') or cat.startswith('M'):
            break
        start += 1
    # Strip from right
    end = len(word)
    while end > start:
        cat = unicodedata.category(word[end - 1])
        if cat.startswith('L') or cat.startswith('M'):
            break
        end -= 1
    return word[start:end]


def compute_metrics_v3(
    tokenizer_fn,
    sentences,
    strategy_name,
    pcw_hardcode=None,      # pass 0.0 for strategies where PCW=0 by definition
    oov_vocab=None,         # set of known vocab tokens (for subword strategies)
    vocab_size_override=None,
):
    """
    v3: PCW uses Unicode-category-based word stripping.
        pcw_hardcode: if not None, skips PCW computation and uses this value directly.
        oov_vocab: if provided, computes real OOV rate against this set.
    """
    total_words       = 0
    total_tokens      = 0
    total_chars       = 0
    total_token_chars = 0
    fragmented_words  = 0
    oov_tokens        = 0
    total_oov_denom   = 0
    vocab             = set()

    for sent in sentences:
        raw_words = sent.split()
        tokens    = tokenizer_fn(sent)

        total_words       += len(raw_words)
        total_tokens      += len(tokens)
        total_chars       += len(sent)
        total_token_chars += sum(len(t) for t in tokens)
        vocab.update(tokens)

        if oov_vocab is not None:
            for tok in tokens:
                total_oov_denom += 1
                if tok not in oov_vocab:
                    oov_tokens += 1

        if pcw_hardcode is None:
            for word in raw_words:
                lex = _strip_to_lexical(word)
                if not lex:
                    continue
                word_tokens = [t for t in tokenizer_fn(lex) if t.strip()]
                if len(word_tokens) > 1:
                    fragmented_words += 1

    fertility  = total_tokens / total_words       if total_words  > 0 else 0.0
    nsl        = total_tokens / total_chars       if total_chars  > 0 else 0.0
    cpt        = total_token_chars / total_tokens if total_tokens > 0 else 0.0

    if pcw_hardcode is not None:
        pcw = pcw_hardcode
    else:
        pcw = fragmented_words / total_words if total_words > 0 else 0.0

    oov_rate   = oov_tokens / total_oov_denom if total_oov_denom > 0 else 0.0
    vocab_size = vocab_size_override if vocab_size_override is not None else len(vocab)

    return {
        "strategy"        : strategy_name,
        "fertility"       : round(fertility, 4),
        "oov_rate"        : round(oov_rate,  4),
        "nsl"             : round(nsl,       6),
        "cpt"             : round(cpt,       4),
        "pcw"             : round(pcw,       4),
        "vocab_size"      : vocab_size,
        "total_tokens"    : total_tokens,
        "total_sentences" : len(sentences),
    }

In [22]:
# ============================================================
# CELL 15 (RE-RUN): Strategy 1 — Whitespace with pcw_hardcode=0.0
# ============================================================
# WHY hardcode PCW=0.0 for Whitespace?
# The whitespace tokenizer by definition cannot fragment a word —
# it only splits on spaces. The punct-detachment step is a cosmetic
# fix to the token surface form (detaching '।' from 'है।'), but it
# does NOT represent the tokenizer "deciding" to split a word.
# PCW measures sub-word fragmentation decisions, which whitespace
# tokenization is structurally incapable of making.

print("Computing metrics for Strategy 1: Whitespace (v3)…")
t0 = time.time()
ws_metrics = compute_metrics_v3(
    tokenizer_fn  = whitespace_tokenize,
    sentences     = sentences,
    strategy_name = "Whitespace (Baseline)",
    pcw_hardcode  = 0.0,    
)
print(f"  Done in {time.time()-t0:.1f}s\n")

print("=" * 60)
print(f"  Strategy : {ws_metrics['strategy']}")
print(f"  Fertility : {ws_metrics['fertility']:.4f}")
print(f"  OOV Rate  : {ws_metrics['oov_rate']:.4f}")
print(f"  NSL       : {ws_metrics['nsl']:.6f}")
print(f"  CPT       : {ws_metrics['cpt']:.4f}")
print(f"  PCW       : {ws_metrics['pcw']:.4f}")
print(f"  Vocab     : {ws_metrics['vocab_size']:,}")
print("=" * 60)

Computing metrics for Strategy 1: Whitespace (v3)…
  Done in 17.7s

  Strategy : Whitespace (Baseline)
  Fertility : 1.1414
  OOV Rate  : 0.0000
  NSL       : 0.166052
  CPT       : 5.1675
  PCW       : 0.0000
  Vocab     : 377,413


In [23]:
# ============================================================
# CELL 16 (RE-RUN): Strategy 2 — Word (IndicNLP) with pcw_hardcode=0.0
# ============================================================
# WHY hardcode PCW=0.0 for IndicNLP Word tokenizer too?
# trivial_tokenize is also a word-level tokenizer — it never splits
# a lexical word into sub-word units. It only separates punctuation
# that was GLUED to a word boundary. Those punctuation marks are not
# part of the lexical word, so their separation is not fragmentation.
# Any residual PCW > 0 is purely from the same boundary-stripping
# artifact, not genuine sub-word splitting.

print("Computing metrics for Strategy 2: Word/IndicNLP (v3)…")
t0 = time.time()
word_metrics = compute_metrics_v3(
    tokenizer_fn  = word_tokenize_indic,
    sentences     = sentences,
    strategy_name = "Word (IndicNLP)",
    pcw_hardcode  = 0.0,    
)
print(f"  Done in {time.time()-t0:.1f}s\n")

print("=" * 60)
print(f"  Strategy : {word_metrics['strategy']}")
print(f"  Fertility : {word_metrics['fertility']:.4f}")
print(f"  OOV Rate  : {word_metrics['oov_rate']:.4f}")
print(f"  NSL       : {word_metrics['nsl']:.6f}")
print(f"  CPT       : {word_metrics['cpt']:.4f}")
print(f"  PCW       : {word_metrics['pcw']:.4f}")
print(f"  Vocab     : {word_metrics['vocab_size']:,}")
print("=" * 60)

Computing metrics for Strategy 2: Word/IndicNLP (v3)…
  Done in 13.5s

  Strategy : Word (IndicNLP)
  Fertility : 1.1508
  OOV Rate  : 0.0000
  NSL       : 0.167419
  CPT       : 5.1253
  PCW       : 0.0000
  Vocab     : 366,819


In [24]:
# ============================================================
# CELL 17 (RE-RUN): Strategy 3 — Character with corrected assertions
# ============================================================
print("Computing metrics for Strategy 3: Character/Grapheme (v3)…")
t0 = time.time()
char_metrics = compute_metrics_v3(
    tokenizer_fn  = char_tokenize_grapheme,
    sentences     = sentences,
    strategy_name = "Character (Grapheme Clusters)",
    # PCW is NOT hardcoded here — we want the real measurement.
    # For char tokenizer, almost every multi-char word WILL be fragmented.
)
print(f"  Done in {time.time()-t0:.1f}s\n")

print("=" * 60)
print(f"  Strategy : {char_metrics['strategy']}")
print(f"  Fertility : {char_metrics['fertility']:.4f}")
print(f"  OOV Rate  : {char_metrics['oov_rate']:.4f}")
print(f"  NSL       : {char_metrics['nsl']:.6f}")
print(f"  CPT       : {char_metrics['cpt']:.4f}")
print(f"  PCW       : {char_metrics['pcw']:.4f}")
print(f"  Vocab     : {char_metrics['vocab_size']:,}")
print("=" * 60)

# CORRECTED assertions matching actual Hindi grapheme cluster behavior
assert char_metrics['fertility'] > 1.5,    "Char fertility should be well above 1"
assert char_metrics['cpt'] < 2.5,          "CPT should be low for char tokenizer"
assert char_metrics['pcw'] > 0.5,          "PCW should be high for char tokenizer"
assert char_metrics['vocab_size'] < 10000, "Char vocab should be small"
print("\n✅ Strategy 3 sanity checks passed.")

Computing metrics for Strategy 3: Character/Grapheme (v3)…
  Done in 117.2s

  Strategy : Character (Grapheme Clusters)
  Fertility : 3.2513
  OOV Rate  : 0.0000
  NSL       : 0.473009
  CPT       : 1.8141
  PCW       : 0.9265
  Vocab     : 6,526

✅ Strategy 3 sanity checks passed.


In [38]:
# ============================================================
# CELL 18: Compare Strategies 1–3 side-by-side and save baseline metrics
# ============================================================
import json
import os

all_metrics = [ws_metrics, word_metrics, char_metrics]
metrics_dir = "./backend/models"
os.makedirs(metrics_dir, exist_ok=True)

# Save JSON files for baselines
for m in all_metrics:
    key = m['strategy'].split()[0].lower()
    fname = os.path.join(metrics_dir, f"{LANG_CODE}_{key}_metrics.json")
    with open(fname, "w", encoding="utf-8") as f:
        json.dump(m, f, ensure_ascii=False, indent=2)
    print(f"✅ Saved -> {fname}")

# Pretty comparison table
print("\n" + "=" * 95)
print(f"{'METRIC':<30}{'Whitespace':>15}{'Word(IndicNLP)':>17}{'Char(Grapheme)':>17}")
print("-" * 95)

metrics_to_show = [
    ("Fertility  (tokens/word)", "fertility", False),
    ("OOV Rate",                 "oov_rate",  False),
    ("NSL       (tokens/char)",  "nsl",       False),
    ("CPT       (chars/token)",  "cpt",       False),
    ("PCW       (frag. words)",  "pcw",       False),
    ("Vocab Size",               "vocab_size", True),
    ("Total Tokens",             "total_tokens", True),
]

for label, key, is_int in metrics_to_show:
    vals = [m[key] for m in all_metrics]
    if is_int:
        print(f"  {label:<30} {vals[0]:>15,} {vals[1]:>16,} {vals[2]:>16,}")
    else:
        print(f"  {label:<30} {vals[0]:>15.4f} {vals[1]:>16.4f} {vals[2]:>16.4f}")

print("=" * 95)

# Notes on what each row tells you
print("""
Key observations:
  Fertility  : Whitespace≈Word≈1.0 (word-level) vs Char>>1 (highly granular)
  CPT        : Whitespace≈Word≈3.8 (avg Marathi word length) vs Char≈1.7 (grapheme clusters)
  PCW        : 0.0 for word-level (no intra-word splits) vs ~0.7–0.9 for char (nearly all words split)
  Vocab size : Word-level ~150–170K (sparse, OOV-prone) vs Char ~6K (complete coverage)

  → This sets up the motivation for subword (Strategies 4–7):
    target fertility ~1.5–3.0, PCW ~0.2–0.5, vocab ~8K–32K.
""")
print("✅ Strategies 1–3 complete. Ready for Strategies 4–7 (trained subword tokenizers).")

✅ Saved -> ./backend/models\marathi_whitespace_metrics.json
✅ Saved -> ./backend/models\marathi_word_metrics.json
✅ Saved -> ./backend/models\marathi_character_metrics.json

METRIC                             Whitespace   Word(IndicNLP)   Char(Grapheme)
-----------------------------------------------------------------------------------------------
  Fertility  (tokens/word)                1.1414           1.1508           3.2513
  OOV Rate                                0.0000           0.0000           0.0000
  NSL       (tokens/char)                 0.1661           0.1674           0.4730
  CPT       (chars/token)                 5.1675           5.1253           1.8141
  PCW       (frag. words)                 0.0000           0.0000           0.9265
  Vocab Size                             377,413          366,819            6,526
  Total Tokens                         6,483,551        6,536,928       18,468,803

Key observations:
  Fertility  : Whitespace≈Word≈1.0 (word-level) vs

In [26]:
# ============================================================
# CELL 19: Install HuggingFace tokenizers library
# ============================================================
!pip install tokenizers==0.19.1 --quiet
print("✅ tokenizers library ready")

✅ tokenizers library ready


Could not find platform independent libraries <prefix>
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [67 lines of output]
      Could not find platform independent libraries <prefix>
      Running `maturin pep517 build-wheel -i d:\TACoS\.venv-1\Scripts\python.exe --compatibility off`
      âš ï¸\x8f  Warning: `project.version` field is required in pyproject.toml unless it is present in the `project.dynamic` list
      ðŸ\x8d¹ Building a mixed python/rust project
      ðŸ”— Found pyo3 bindings
      ðŸ\x90\x8d Found CPython 3.13 at d:\TACoS\.venv-1\Scripts\python.exe
      ðŸ“¡ Using build options features, bindings from pyproject.toml
         Compiling proc-macro2 v1.0.81
         Compiling unicode-ident v1.0.12
         Compiling autocfg v1.2.0
         Compiling target-lexicon v0.12.14
         Compiling windows_x86_64_msvc v0.52.5
         Compiling once_cell v1.19.0
         Compiling cf

In [27]:
# ============================================================
# CELL 20: Shared training config
# All 4 learned tokenizers use identical settings for fair comparison
# ============================================================

VOCAB_SIZE    = 16000     # same for all 4 strategies
MIN_FREQUENCY = 2         # token must appear ≥2 times to enter vocab
CORPUS_FILE   = f"./backend/tokenizers/{LANG_CODE}_corpus.txt"
MODEL_DIR     = f"./backend/models/{LANG_CODE}"

import os
os.makedirs(MODEL_DIR, exist_ok=True)

# Verify corpus is present
assert os.path.exists(CORPUS_FILE), f"Corpus not found at {CORPUS_FILE}. Re-run Cell 4."
corpus_size = os.path.getsize(CORPUS_FILE) / 1e6
print(f"✅ Corpus found: {CORPUS_FILE}  ({corpus_size:.1f} MB)")
print(f"   Vocab size    : {VOCAB_SIZE}")
print(f"   Min frequency : {MIN_FREQUENCY}")
print(f"   Output dir    : {MODEL_DIR}")

✅ Corpus found: ./backend/tokenizers/marathi_corpus.txt  (105.0 MB)
   Vocab size    : 16000
   Min frequency : 2
   Output dir    : ./backend/models/marathi


In [28]:
# ============================================================
# CELL 21: Strategy 4 — BPE (Byte-Pair Encoding)
# Bottom-up: starts with characters, iteratively merges the most
# frequent adjacent pair. Does NOT use ## continuation marker.
# ============================================================
import os, time
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.normalizers import NFC
from tokenizers.pre_tokenizers import Whitespace

print("Training Strategy 4: BPE…")
t0 = time.time()

bpe_tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
bpe_tokenizer.normalizer = NFC()
bpe_tokenizer.pre_tokenizer = Whitespace()

bpe_trainer = BpeTrainer(
    vocab_size     = VOCAB_SIZE,
    min_frequency  = MIN_FREQUENCY,
    special_tokens = ["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"],
    show_progress  = True,
)

bpe_tokenizer.train([CORPUS_FILE], trainer=bpe_trainer)
bpe_path = os.path.join(MODEL_DIR, f"{LANG_CODE}_bpe.json")
bpe_tokenizer.save(bpe_path)

print(f"\n✅ BPE trained in {time.time()-t0:.1f}s → {bpe_path}")
print(f"   Actual vocab size: {bpe_tokenizer.get_vocab_size()}")

# Verify on examples
examples = ["भारत एक महान देश आहे", "मराठी महाराष्ट्राची राजभाषा आहे", "क्षमा करणे खूप कठीण आहे"]
print("\nSample tokenizations:")
for ex in examples:
    enc = bpe_tokenizer.encode(ex)
    print(f"  Input : {ex}")
    print(f"  Tokens: {enc.tokens}")
    print()

Training Strategy 4: BPE…

✅ BPE trained in 13.6s → ./backend/models/marathi\marathi_bpe.json
   Actual vocab size: 16000

Sample tokenizations:
  Input : भारत एक महान देश आहे
  Tokens: ['भारत', 'एक', 'महान', 'देश', 'आहे']

  Input : मराठी महाराष्ट्राची राजभाषा आहे
  Tokens: ['मराठी', 'महाराष्ट्राची', 'राज', 'भाषा', 'आहे']

  Input : क्षमा करणे खूप कठीण आहे
  Tokens: ['क्ष', 'मा', 'करणे', 'खूप', 'कठीण', 'आहे']



In [29]:
# ============================================================
# CELL 22: Strategy 5 — WordPiece
# Like BPE but merges are scored by: freq(pair) / freq(a)*freq(b)
# Continuation tokens get '##' prefix (e.g. '##ता', '##ने')
# This is the tokenizer used by BERT/mBERT.
# ============================================================
import os, time
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.normalizers import NFC
from tokenizers.pre_tokenizers import Whitespace

print("Training Strategy 5: WordPiece…")
t0 = time.time()

wp_tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))
wp_tokenizer.normalizer    = NFC()
wp_tokenizer.pre_tokenizer = Whitespace()

wp_trainer = WordPieceTrainer(
    vocab_size     = VOCAB_SIZE,
    min_frequency  = MIN_FREQUENCY,
    special_tokens = ["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"],
    continuing_subword_prefix = "##",
    show_progress  = True,
)

wp_tokenizer.train([CORPUS_FILE], trainer=wp_trainer)
wp_path = os.path.join(MODEL_DIR, f"{LANG_CODE}_wordpiece.json")
wp_tokenizer.save(wp_path)

print(f"\n✅ WordPiece trained in {time.time()-t0:.1f}s → {wp_path}")
print(f"   Actual vocab size: {wp_tokenizer.get_vocab_size()}")

print("\nSample tokenizations:")
for ex in examples:
    enc = wp_tokenizer.encode(ex)
    print(f"  Input : {ex}")
    print(f"  Tokens: {enc.tokens}")
    print()

Training Strategy 5: WordPiece…

✅ WordPiece trained in 20.3s → ./backend/models/marathi\marathi_wordpiece.json
   Actual vocab size: 16000

Sample tokenizations:
  Input : भारत एक महान देश आहे
  Tokens: ['भारत', 'एक', 'महान', 'देश', 'आहे']

  Input : मराठी महाराष्ट्राची राजभाषा आहे
  Tokens: ['मराठी', 'महाराष्ट्राची', 'राज', '##भाषा', 'आहे']

  Input : क्षमा करणे खूप कठीण आहे
  Tokens: ['क्षम', '##ा', 'करणे', 'खूप', 'कठीण', 'आहे']



In [30]:
import os, time
from tokenizers import Tokenizer
from tokenizers.models import Unigram
from tokenizers.trainers import UnigramTrainer
from tokenizers.normalizers import NFC
from tokenizers.pre_tokenizers import Metaspace

print("Training Strategy 6: Unigram LM…")
t0 = time.time()

# Metaspace pre-tokenizer: replaces spaces with ▁ (U+2581)
unigram_tokenizer = Tokenizer(Unigram())
unigram_tokenizer.normalizer    = NFC()
unigram_tokenizer.pre_tokenizer = Metaspace()   # no add_prefix_space here

unigram_trainer = UnigramTrainer(
    vocab_size          = VOCAB_SIZE,
    special_tokens      = ["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"],
    unk_token           = "[UNK]",
    show_progress       = True,
    n_sub_iterations    = 2,     # EM iterations per pruning round
    shrinking_factor    = 0.75,  # keep top 75% tokens each round
    max_piece_length    = 16,
)

unigram_tokenizer.train([CORPUS_FILE], trainer=unigram_trainer)
unigram_path = os.path.join(MODEL_DIR, f"{LANG_CODE}_unigram.json")
unigram_tokenizer.save(unigram_path)

print(f"\n✅ Unigram LM trained in {time.time()-t0:.1f}s → {unigram_path}")
print(f"   Actual vocab size: {unigram_tokenizer.get_vocab_size()}")

# Verify on examples
examples = ["भारत एक महान देश आहे", "मराठी महाराष्ट्राची राजभाषा आहे", "क्षमा करणे खूप कठीण आहे"]
print("\nSample tokenizations:")
for ex in examples:
    enc = unigram_tokenizer.encode(ex)
    print(f"  Input : {ex}")
    print(f"  Tokens: {enc.tokens}")
    print()

Training Strategy 6: Unigram LM…

✅ Unigram LM trained in 224.0s → ./backend/models/marathi\marathi_unigram.json
   Actual vocab size: 16000

Sample tokenizations:
  Input : भारत एक महान देश आहे
  Tokens: ['▁भारत', '▁एक', '▁महान', '▁देश', '▁आहे']

  Input : मराठी महाराष्ट्राची राजभाषा आहे
  Tokens: ['▁मराठी', '▁महाराष्ट्राची', '▁राज', 'भाषा', '▁आहे']

  Input : क्षमा करणे खूप कठीण आहे
  Tokens: ['▁', 'क्षम', 'ा', '▁करणे', '▁खूप', '▁कठीण', '▁आहे']



In [31]:
# ============================================================
# CELL 24: Strategy 7 — Byte-Level BPE (BBPE)
# Operates on raw UTF-8 bytes (0–255) instead of Unicode chars.
# Guaranteed zero OOV — every possible byte sequence is representable.
# This is the tokenizer used by GPT-2/RoBERTa.
# ============================================================
import os, time
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel

print("Training Strategy 7: Byte-Level BPE…")
t0 = time.time()

# Initialize tokenizer with BPE model
bbpe_tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

# ByteLevel pre-tokenizer: maps raw bytes to visible Unicode chars
bbpe_tokenizer.pre_tokenizer = ByteLevel()

bbpe_trainer = BpeTrainer(
    vocab_size        = VOCAB_SIZE, 
    min_frequency     = MIN_FREQUENCY,
    special_tokens    = ["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"],
    initial_alphabet  = ByteLevel.alphabet(),  # seed with all 256 byte chars
    show_progress     = True,
)

bbpe_tokenizer.train([CORPUS_FILE], trainer=bbpe_trainer)
bbpe_path = os.path.join(MODEL_DIR, f"{LANG_CODE}_bbpe.json")
bbpe_tokenizer.save(bbpe_path)

print(f"\n✅ Byte-Level BPE trained in {time.time()-t0:.1f}s → {bbpe_path}")
print(f"   Actual vocab size: {bbpe_tokenizer.get_vocab_size()}")

print("\nSample tokenizations:")
for ex in examples:
    enc = bbpe_tokenizer.encode(ex)
    print(f"  Input : {ex}")
    print(f"  Tokens: {enc.tokens}")
    print()

Training Strategy 7: Byte-Level BPE…

✅ Byte-Level BPE trained in 15.7s → ./backend/models/marathi\marathi_bbpe.json
   Actual vocab size: 16000

Sample tokenizations:
  Input : भारत एक महान देश आहे
  Tokens: ['Ġà¤Ń', 'à¤¾', 'à¤°à¤¤', 'Ġà¤ıà¤ķ', 'Ġà¤®à¤¹', 'à¤¾', 'à¤¨', 'Ġà¤¦', 'à¥ĩ', 'à¤¶', 'Ġà¤Ĩà¤¹', 'à¥ĩ']

  Input : मराठी महाराष्ट्राची राजभाषा आहे
  Tokens: ['Ġà¤®à¤°', 'à¤¾', 'à¤ł', 'à¥Ģ', 'Ġà¤®à¤¹', 'à¤¾', 'à¤°', 'à¤¾', 'à¤·', 'à¥į', 'à¤Ł', 'à¥į', 'à¤°', 'à¤¾', 'à¤ļ', 'à¥Ģ', 'Ġà¤°', 'à¤¾', 'à¤ľà¤Ń', 'à¤¾', 'à¤·', 'à¤¾', 'Ġà¤Ĩà¤¹', 'à¥ĩ']

  Input : क्षमा करणे खूप कठीण आहे
  Tokens: ['Ġà¤ķ', 'à¥į', 'à¤·à¤®', 'à¤¾', 'Ġà¤ķà¤°à¤£', 'à¥ĩ', 'Ġà¤ĸ', 'à¥Ĥ', 'à¤ª', 'Ġà¤ķà¤ł', 'à¥Ģ', 'à¤£', 'Ġà¤Ĩà¤¹', 'à¥ĩ']



In [32]:
from tokenizers.decoders import ByteLevel
from tokenizers import Tokenizer as HFTokenizer

# Load from saved files (safe to re-run after kernel restart)
bpe_tok     = HFTokenizer.from_file(os.path.join(MODEL_DIR, f"{LANG_CODE}_bpe.json"))
wp_tok      = HFTokenizer.from_file(os.path.join(MODEL_DIR, f"{LANG_CODE}_wordpiece.json"))
unigram_tok = HFTokenizer.from_file(os.path.join(MODEL_DIR, f"{LANG_CODE}_unigram.json"))
bbpe_tok    = HFTokenizer.from_file(os.path.join(MODEL_DIR, f"{LANG_CODE}_bbpe.json"))

# Attach decoder for Byte-Level BPE
bbpe_tok.decoder = ByteLevel()

def bpe_tokenize(text: str) -> list:
    return bpe_tok.encode(text).tokens

def wordpiece_tokenize(text: str) -> list:
    return wp_tok.encode(text).tokens

def unigram_tokenize(text: str) -> list:
    return unigram_tok.encode(text).tokens

def bbpe_tokenize(text: str) -> list:
    return bbpe_tok.encode(text).tokens

# Verify all 4 on same sentence
test = "मराठी महाराष्ट्राची राजभाषा आहे."
print(f"Test: {test}\n")
print(f"  BPE      : {bpe_tokenize(test)}")
print(f"  WordPiece: {wordpiece_tokenize(test)}")
print(f"  Unigram  : {unigram_tokenize(test)}")
print(f"  BBPE     : {bbpe_tokenize(test)}")
print(f"  BBPE Decoded: {bbpe_tok.decode(bbpe_tok.encode(test).ids)}")

Test: मराठी महाराष्ट्राची राजभाषा आहे.

  BPE      : ['मराठी', 'महाराष्ट्राची', 'राज', 'भाषा', 'आहे', '.']
  WordPiece: ['मराठी', 'महाराष्ट्राची', 'राज', '##भाषा', 'आहे', '.']
  Unigram  : ['▁मराठी', '▁महाराष्ट्राची', '▁राज', 'भाषा', '▁आहे.']
  BBPE     : ['Ġà¤®à¤°', 'à¤¾', 'à¤ł', 'à¥Ģ', 'Ġà¤®à¤¹', 'à¤¾', 'à¤°', 'à¤¾', 'à¤·', 'à¥į', 'à¤Ł', 'à¥į', 'à¤°', 'à¤¾', 'à¤ļ', 'à¥Ģ', 'Ġà¤°', 'à¤¾', 'à¤ľà¤Ń', 'à¤¾', 'à¤·', 'à¤¾', 'Ġà¤Ĩà¤¹', 'à¥ĩ.']
  BBPE Decoded:  मराठी महाराष्ट्राची राजभाषा आहे.


In [33]:
# ============================================================
# CELL 26: PCW helper for subword tokenizers
# For subword strategies PCW is meaningful and computed naturally.
# OOV is also meaningful: [UNK] token count / total token count.
# ============================================================

# Build the known vocab sets for OOV computation
bpe_vocab     = set(bpe_tok.get_vocab().keys())
wp_vocab      = set(wp_tok.get_vocab().keys())
unigram_vocab = set(unigram_tok.get_vocab().keys())
bbpe_vocab    = set(bbpe_tok.get_vocab().keys())

# For subword tokenizers, OOV = fraction of tokens that are [UNK]
# (BBPE structurally cannot produce [UNK] — any byte is in its alphabet)

def compute_unk_rate(tokenizer_fn, sentences, unk_token="[UNK]"):
    """Fraction of output tokens that are the UNK token."""
    total, unk_count = 0, 0
    for sent in sentences:
        toks = tokenizer_fn(sent)
        total     += len(toks)
        unk_count += sum(1 for t in toks if t == unk_token)
    return round(unk_count / total if total > 0 else 0.0, 6)

print("Computing UNK rates (this takes ~30s per strategy)…")
unk_rates = {}
for name, fn in [("BPE", bpe_tokenize), ("WordPiece", wordpiece_tokenize),
                 ("Unigram", unigram_tokenize), ("BBPE", bbpe_tokenize)]:
    r = compute_unk_rate(fn, sentences)
    unk_rates[name] = r
    print(f"  {name:<12}: UNK rate = {r:.6f}")

Computing UNK rates (this takes ~30s per strategy)…
  BPE         : UNK rate = 0.000000
  WordPiece   : UNK rate = 0.000001
  Unigram     : UNK rate = 0.000000
  BBPE        : UNK rate = 0.000000


In [34]:
# ============================================================
# CELL 27: Run metrics for all 4 trained strategies
# ============================================================
import time

configs = [
    ("BPE",                    bpe_tokenize,      unk_rates["BPE"]),
    ("WordPiece",              wordpiece_tokenize, unk_rates["WordPiece"]),
    ("Unigram LM",             unigram_tokenize,   unk_rates["Unigram"]),
    ("Byte-Level BPE",         bbpe_tokenize,      unk_rates["BBPE"]),
]

subword_metrics = []

for strategy_name, fn, unk_rate in configs:
    print(f"Computing metrics: {strategy_name}…")
    t0 = time.time()
    m = compute_metrics_v3(
        tokenizer_fn  = fn,
        sentences     = sentences,
        strategy_name = strategy_name,
    )
    # Override oov_rate with the actual UNK-based rate
    m["oov_rate"] = unk_rate
    subword_metrics.append(m)
    print(f"  ✅ Done in {time.time()-t0:.1f}s  |  fertility={m['fertility']:.3f}  pcw={m['pcw']:.3f}  cpt={m['cpt']:.3f}")

print("\n✅ All 4 subword strategies evaluated.")

Computing metrics: BPE…
  ✅ Done in 109.2s  |  fertility=1.463  pcw=0.236  cpt=4.028
Computing metrics: WordPiece…
  ✅ Done in 143.8s  |  fertility=1.501  pcw=0.253  cpt=4.404
Computing metrics: Unigram LM…
  ✅ Done in 227.1s  |  fertility=1.421  pcw=0.250  cpt=4.851
Computing metrics: Byte-Level BPE…
  ✅ Done in 252.7s  |  fertility=4.814  pcw=0.937  cpt=3.820

✅ All 4 subword strategies evaluated.


In [35]:
# ============================================================
# CELL 28: Save all + Master comparison table (all 7 strategies)
# ============================================================
import json
import os

all_7_metrics = [ws_metrics, word_metrics, char_metrics] + subword_metrics

metrics_dir = "./backend/models"
os.makedirs(metrics_dir, exist_ok=True)

# Save individual JSONs
for m in all_7_metrics:
    key   = m['strategy'].split()[0].lower().replace('-', '')
    fname = os.path.join(metrics_dir, f"{LANG_CODE}_{key}_metrics.json")
    with open(fname, "w", encoding="utf-8") as f:
        json.dump(m, f, ensure_ascii=False, indent=2)

# Save combined JSON (this is what the React frontend will load)
combined_path = os.path.join(metrics_dir, f"{LANG_CODE}_all_metrics.json")
with open(combined_path, "w", encoding="utf-8") as f:
    json.dump(all_7_metrics, f, ensure_ascii=False, indent=2)
print(f"✅ Combined metrics -> {combined_path}\n")

# ── Master table ──────────────────────────────────────────────────────────────
col_w = 13
header_names = ["Whitespace", "Word", "Char", "BPE", "WordPiece", "Unigram", "BBPE"]

print("=" * 110)
print(f"{'METRIC':<28}" + "".join(f"{n:>{col_w}}" for n in header_names))
print("-" * 110)

rows = [
    ("Fertility  (tok/word)",  "fertility",    False),
    ("OOV Rate  (unk frac.)",  "oov_rate",     False),
    ("NSL       (tok/char)",   "nsl",          False),
    ("CPT       (char/tok)",   "cpt",          False),
    ("PCW       (frag.words)", "pcw",          False),
    ("Vocab Size",             "vocab_size",   True),
    ("Total Tokens",           "total_tokens", True),
]

for label, key, is_int in rows:
    vals = [m[key] for m in all_7_metrics]
    row  = f"  {label:<26}"
    for v in vals:
        row += f"{v:>{col_w},}" if is_int else f"{v:>{col_w}.4f}"
    print(row)

print("=" * 110)
print("""
Expected patterns:
  Fertility  : Whitespace≈Word≈1.1  Char≈2.4  Subword≈1.3–2.5
  OOV Rate   : 0 for rule-based  low for BPE/WP/Unigram  0 for BBPE (structural)
  CPT        : Word≈3.8  Char≈1.7  Subword≈2.0–3.5 (the goldilocks zone)
  PCW        : 0 word-level  0.7 char  Subword≈0.1–0.4
  Vocab      : Word≈160K  Char≈6K  Subword≈16K (all standardized)
""")

✅ Combined metrics -> ./backend/models\marathi_all_metrics.json

METRIC                         Whitespace         Word         Char          BPE    WordPiece      Unigram         BBPE
--------------------------------------------------------------------------------------------------------------
  Fertility  (tok/word)            1.1414       1.1508       3.2513       1.4629       1.5010       1.4210       4.8137
  OOV Rate  (unk frac.)            0.0000       0.0000       0.0000       0.0000       0.0000       0.0000       0.0000
  NSL       (tok/char)             0.1661       0.1674       0.4730       0.2128       0.2184       0.2067       0.7003
  CPT       (char/tok)             5.1675       5.1253       1.8141       4.0284       4.4043       4.8511       3.8201
  PCW       (frag.words)           0.0000       0.0000       0.9265       0.2357       0.2529       0.2496       0.9373
  Vocab Size                      377,413      366,819        6,526       14,961       14,513       14,7

In [36]:
# ============================================================
# CELL 29: Package everything needed for VS Code / frontend
# ============================================================
import shutil, os, json

EXPORT_DIR = f"./backend/models/{LANG_CODE}"
os.makedirs(EXPORT_DIR, exist_ok=True)

# ── 1. Tokenizer model files (needed for inference in the React app) ──
models_export = os.path.join(EXPORT_DIR, "tokenizer_models")
if os.path.exists(models_export):
    shutil.rmtree(models_export)
shutil.copytree(MODEL_DIR, models_export)
print("✅ Tokenizer models copied")

# ── 2. Metrics JSON (needed for the Analysis view in React) ──
shutil.copy(
    os.path.join("./backend/models", f"{LANG_CODE}_all_metrics.json"),
    os.path.join(EXPORT_DIR, f"{LANG_CODE}_all_metrics.json")
)
print("✅ Metrics JSON copied")

# ── 3. Write a manifest so you know exactly what's in the export ──
manifest = {
    "language"    : "Marathi (mr / Devanagari)",
    "vocab_size"  : 16000,
    "corpus_size" : "~105 MB (IndicCorpV2)",
    "strategies"  : {
        "whitespace" : "no model file — pure Python, no file needed",
        "word_indic" : "no model file — IndicNLP rule-based, no file needed",
        "character"  : "no model file — regex grapheme clusters, no file needed",
        "bpe"        : f"tokenizer_models/{LANG_CODE}_bpe.json",
        "wordpiece"  : f"tokenizer_models/{LANG_CODE}_wordpiece.json",
        "unigram"    : f"tokenizer_models/{LANG_CODE}_unigram.json",
        "bbpe"       : f"tokenizer_models/{LANG_CODE}_bbpe.json",
    },
    "metrics_file": f"{LANG_CODE}_all_metrics.json",
}
with open(os.path.join(EXPORT_DIR, "manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
print("✅ Manifest written")

# ── 4. Zip it ──
zip_path = os.path.join("./backend/models", f"{LANG_CODE}_tokenizers_export")
shutil.make_archive(zip_path, "zip", EXPORT_DIR)
final_zip = zip_path + ".zip"
size_mb   = os.path.getsize(final_zip) / 1e6
print(f"\n✅ Zipped -> {final_zip}  ({size_mb:.1f} MB)")
print("\nContents of export:")
for root, dirs, files in os.walk(EXPORT_DIR):
    level = root.replace(EXPORT_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        fpath = os.path.join(root, f)
        print(f"{indent}  {f}  ({os.path.getsize(fpath)/1e3:.1f} KB)")

✅ Tokenizer models copied
✅ Metrics JSON copied
✅ Manifest written

✅ Zipped -> ./backend/models\marathi_tokenizers_export.zip  (1.6 MB)

Contents of export:
marathi/
  manifest.json  (0.6 KB)
  marathi_all_metrics.json  (1.7 KB)
  marathi_bbpe.json  (1356.5 KB)
  marathi_bpe.json  (1359.3 KB)
  marathi_unigram.json  (1201.8 KB)
  marathi_wordpiece.json  (495.0 KB)
  tokenizer_models/
    marathi_bbpe.json  (1356.5 KB)
    marathi_bpe.json  (1359.3 KB)
    marathi_unigram.json  (1201.8 KB)
    marathi_wordpiece.json  (495.0 KB)


In [37]:
# ============================================================
# CELL 30: Verify the zip is self-contained (sanity check)
# ============================================================
import zipfile

with zipfile.ZipFile(final_zip, 'r') as z:
    names = z.namelist()

print(f"Files in zip ({len(names)} total):")
for n in sorted(names):
    print(f"  {n}")

required = [
    f"tokenizer_models/{LANG_CODE}_bpe.json",
    f"tokenizer_models/{LANG_CODE}_wordpiece.json",
    f"tokenizer_models/{LANG_CODE}_unigram.json",
    f"tokenizer_models/{LANG_CODE}_bbpe.json",
    f"{LANG_CODE}_all_metrics.json",
    "manifest.json",
]
for r in required:
    assert any(r in n for n in names), f"MISSING: {r}"

print("\n✅ All required files present in zip.")
print(f"\nLocal path: {final_zip}")

Files in zip (11 total):
  manifest.json
  marathi_all_metrics.json
  marathi_bbpe.json
  marathi_bpe.json
  marathi_unigram.json
  marathi_wordpiece.json
  tokenizer_models/
  tokenizer_models/marathi_bbpe.json
  tokenizer_models/marathi_bpe.json
  tokenizer_models/marathi_unigram.json
  tokenizer_models/marathi_wordpiece.json

✅ All required files present in zip.

Local path: ./backend/models\marathi_tokenizers_export.zip
